<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.7-graphrag/notebooks/exercises-6.7.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.7 — GraphRAG
Netsetos GenAI Course v2.0 | Module 6

Knowledge graphs, Microsoft GraphRAG, LightRAG, PropertyGraphIndex, production patterns


In [ ]:
# pip install graphrag lightrag-hku llama-index-graph-stores-neo4j neo4j networkx -q
print('Ready for GraphRAG')


## Ex 1: Build KG from Scratch


In [ ]:
import networkx as nx
import json

# LLM-extracted entities and relationships
triples = [
    ('GraphRAG', 'USES', 'Knowledge Graphs'),
    ('GraphRAG', 'DEVELOPED_BY', 'Microsoft'),
    ('Knowledge Graphs', 'CONTAIN', 'Entities'),
    ('Knowledge Graphs', 'CONTAIN', 'Relationships'),
    ('Leiden Algorithm', 'DETECTS', 'Communities'),
    ('GraphRAG', 'USES', 'Leiden Algorithm'),
    ('Communities', 'HAVE', 'LLM Summaries'),
    ('Global Search', 'QUERIES', 'LLM Summaries'),
    ('LightRAG', 'ALTERNATIVE_TO', 'GraphRAG'),
    ('LightRAG', 'USES', 'Dual Level Retrieval'),
]

G = nx.DiGraph()
for s, r, o in triples:
    G.add_edge(s, o, relation=r)

print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}')
for node in ['GraphRAG', 'Knowledge Graphs']:
    neighbors = list(G.neighbors(node))
    print(f'{node} → {neighbors}')


## Ex 2: Microsoft GraphRAG Quickstart


In [ ]:
# pip install graphrag
# graphrag init --root ./my_project
# # Edit .env: GRAPHRAG_API_KEY=sk-...
# graphrag index                          # Standard (LLM extraction)
# graphrag index --method fast            # NLP extraction (much cheaper)
# graphrag query 'Who is Scrooge?' --method local
# graphrag query 'What are the main themes?' --method global

print('Microsoft GraphRAG CLI:')
print('  graphrag init → creates settings.yaml + .env')
print('  graphrag index → 6-phase pipeline')
print('  graphrag index --method fast → NLP extraction (cheap)')
print('  graphrag query --method local → entity-centric')
print('  graphrag query --method global → map-reduce over communities')
print('  graphrag query --method drift → community → sub-questions → local')
print()
print('Cost: Wizard of Oz (~38K tokens)')
print('  GPT-4o-mini: $0.34 | GPT-4o: $1.70 | GPT-4-Turbo: $3.40')


## Ex 3: LightRAG (6,000× Cheaper)


In [ ]:
# from lightrag import LightRAG, QueryParam
# rag = LightRAG(working_dir='./lightrag_data')
# rag.insert('Your document text here...')
# # Dual-level retrieval
# result = rag.query('What are the main themes?', param=QueryParam(mode='hybrid'))

print('LightRAG modes:')
print('  naive: standard vector retrieval')
print('  local: entity-level (specific facts)')
print('  global: topic-level (broader themes)')
print('  hybrid: both entity + topic (recommended)')
print()
print('Cost comparison (A Christmas Carol):')
print('  Microsoft GraphRAG: $6-7')
print('  LightRAG: ~$0.15 (40× cheaper)')
print('  Retrieval tokens: GraphRAG ~610K vs LightRAG <100')


## Ex 4: LlamaIndex PropertyGraphIndex


In [ ]:
from typing import Literal

# Schema-guided extraction
entities = Literal['PERSON', 'ORGANIZATION', 'CONCEPT', 'TECHNOLOGY']
relations = Literal['USES', 'DEVELOPED_BY', 'ALTERNATIVE_TO', 'PART_OF']

# from llama_index.core.indices.property_graph import SchemaLLMPathExtractor
# extractor = SchemaLLMPathExtractor(
#     llm=llm, possible_entities=entities,
#     possible_relations=relations, strict=True
# )
# from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
# graph_store = Neo4jPropertyGraphStore(url='bolt://localhost:7687')
# index = PropertyGraphIndex.from_documents(
#     docs, property_graph_store=graph_store,
#     kg_extractors=[extractor]
# )

print('PropertyGraphIndex extractors:')
print('  SchemaLLMPathExtractor: constrained (production)')
print('  SimpleLLMPathExtractor: open discovery')
print('  ImplicitPathExtractor: structural paths')
print('\nRetrievers:')
for r in ['LLMSynonymRetriever','VectorContextRetriever','TextToCypherRetriever','CypherTemplateRetriever']:
    print(f'  {r}')


## Ex 5: Neo4j + Cypher Queries


In [ ]:
# from neo4j import GraphDatabase
# driver = GraphDatabase.driver('bolt://localhost:7687', auth=('neo4j','password'))

# Cypher examples:
cypher_queries = [
    ('Find all entities connected to GraphRAG',
     'MATCH (n)-[r]->(m) WHERE n.name = "GraphRAG" RETURN n, r, m'),
    ('Shortest path between two entities',
     'MATCH p = shortestPath((a)-[*]-(b)) WHERE a.name="GraphRAG" AND b.name="Communities" RETURN p'),
    ('Most connected entities (hub nodes)',
     'MATCH (n)-[r]-() RETURN n.name, count(r) as degree ORDER BY degree DESC LIMIT 10'),
]
for desc, query in cypher_queries:
    print(f'{desc}:')
    print(f'  {query}')
    print()


## Ex 6: GraphRAG Evaluation


In [ ]:
print('GraphRAG Evaluation Framework:')
print()
print('Microsoft 4 metrics (LLM-as-judge pairwise):')
for metric, desc in [
    ('Comprehensiveness', 'Coverage of all query aspects'),
    ('Diversity', 'Variety of perspectives'),
    ('Empowerment', 'Helps user reach informed understanding'),
    ('Directness', 'Conciseness of response')]:
    print(f'  {metric:20s}: {desc}')
print()
print('Practical approach:')
print('  1. Create 30-50 test Qs (10 simple + 10 multi-hop + 10 thematic)')
print('  2. Run same Qs on: vector RAG, LightRAG, GraphRAG')
print('  3. LLM-as-judge pairwise comparison')
print('  4. Evaluate retrieval precision separately from generation')
print()
print('Key finding (GraphRAG-Bench ICLR 2026):')
print('  GraphRAG can UNDERPERFORM on simple fact retrieval')
print('  → This is why hybrid routing matters')


## Ex 7: Hybrid Vector + Graph Router


In [ ]:
def classify_query(query: str) -> str:
    '''Simple query classifier for routing'''
    q = query.lower()
    thematic = ['theme', 'summary', 'overview', 'main', 'across all', 'overall']
    relational = ['connected', 'related', 'relationship', 'between', 'compare']
    if any(w in q for w in thematic):
        return 'graphrag_global'
    if any(w in q for w in relational):
        return 'graphrag_local'
    return 'vector_rag'

test_queries = [
    'What is the refund policy?',
    'What are the main themes across all documents?',
    'How is entity A related to entity B?',
]
for q in test_queries:
    route = classify_query(q)
    print(f'  "{q}" → {route}')


## Ex 8: India Legal GraphRAG


In [ ]:
print('India Legal GraphRAG Stack:')
print()
print('Entity Extraction:')
print('  OpenNyAI en_legal_ner_trf: 14 legal entity types')
print('    Court, Judge, Lawyer, Petitioner, Respondent,')
print('    Statute, Provision, Precedent, Date, Organization...')
print()
print('Embeddings:')
print('  InLegalBERT: 5.4M Indian legal docs (27GB)')
print('  1.8M+ HuggingFace downloads')
print()
print('Hindi NER:')
print('  MuRIL + CRF: F1 85.77 (+10% vs mBERT)')
print('  IndicNER (AI4Bharat): 11 Indic languages')
print('  TriNER: F1 92.11 (Hindi + Bengali + Marathi)')
print()
print('Graph: Neo4j + NyOn ontology + NyayGraph (IPC)')
print('Cross-lingual: IndicTrans2 (22 languages)')


---
## Recap
GraphRAG solves what vector RAG cannot: corpus-wide thematic understanding via community summaries. Microsoft GraphRAG: 6-phase pipeline (extract → Leiden communities → LLM summaries), 4 search modes (Local/Global/DRIFT/Basic), 70-80% comprehensiveness win rate, expensive indexing (~75% cost in entity extraction). Alternatives: LightRAG (6,000× fewer retrieval tokens, EMNLP 2025), LazyGraphRAG (0.1% indexing cost), HippoRAG (Personalized PageRank, +20% Recall@5). LlamaIndex PropertyGraphIndex: schema-guided extraction, 4 retriever types, Neo4j integration. Construction: hybrid extraction (GLiNER + LLM), schema-first for production, entity resolution critical (22% duplicates). Production: hybrid routing (vector for simple, graph for relational/thematic), Neo4j default, FalkorDB for AI-first. India: OpenNyAI + InLegalBERT + MuRIL + NyOn + Neo4j.
